# 🌍Small Landscape Features Geospatial Zonal & Temporal Statistics

**Comparing Woody Vegetation Layers (2018 vs 2021) over user-defined Areas of Interest**

---

This notebook guides you through three self-contained geospatial workflows:

| # | Use Case | AOI | Goal |
|---|----------|-----|------|
| A | 🌾 Agriculture | Agricultural parcels | Monitor crop/land indicators per parcel across years |
| B | 🦋 Biodiversity | Administrative units | Aggregate habitat indicators at admin level |
| C | 🏙️ Urban | Urban extents | Track tree canopy extent  |

**Each use case follows the same workflow:**
1. Load & visualise data
2. Select Areas of Interest interactively
3. Run zonal statistics (2018 & 2021)
4. Compute change metrics
5. Visualise results (maps + charts)
6. Export outputs (CSV + GeoJSON)

---
> 📂 **Before running**: Update the file paths in the **Configuration** cell at the beginning of your use case.

## 📋 Table of Contents
- [📦 0 · Imports & Helper Functions](#imports)
- [🌾 Use Case A – Agriculture](#usecaseA)
- [🦋 Use Case B – Biodiversity](#usecaseB)
- [🏙️ Use Case C – Urban](#usecaseC)

### To be improved
Helper functions are (for now 16/04/2026) used by each use case. 
They need to be run after configuration step.
Either the configuration step take into account all use cases and structure can stay as it is right now, or one config per use case at beginning of use case section. If so, the function cell should be duplicate also for each use case

<a id='imports'></a>
## 📦 0 · Imports & Helper Functions

In [ ]:
# =============================================================
#  CONFIGURATION  –  edit this cell before running the notebook
# =============================================================

from pathlib import Path

# ── Root data directory (relative to this notebook) ──────────
DATA_DIR  = Path('data')          # <-- adjust if needed
OUT_DIR   = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Raster inputs ─────────────────────────────────────────────
RASTER_2018 = DATA_DIR / 'rasters' / 'wvl18_lu_subset.tif'   # <-- path to 2018 raster
RASTER_2021 = DATA_DIR / 'rasters' / 'wvl21_lu_subset.tif'   # <-- path to 2021 raster

# ── Vector inputs ─────────────────────────────────────────────
AGRI_VECTOR  = DATA_DIR / 'vectors' / 'lpis_lu_2025_subset.gpkg'
ADMIN_VECTOR = DATA_DIR / 'vectors' / 'lpis_lu_2025_subset2.gpkg'
URBAN_VECTOR = DATA_DIR / 'vectors' / 'lpis_lu_2025_subset3.gpkg'

# ── Raster band to analyse (1-indexed) ────────────────────────
# Change if your raster has multiple bands
BAND_INDEX = 1

# ── Column used as unique identifier in each vector layer ─────
AGRI_ID_COL  = 'FLIK'       # <-- adjust to your attribute name
ADMIN_ID_COL  = 'FLIK'  
URBAN_ID_COL  = 'FLIK'  

# ── Statistics to compute ─────────────────────────────────────
ZONAL_STATS = ['min', 'max', 'mean', 'median', 'std', 'count', 'nodata']

# ── Map visualisation ─────────────────────────────────────────
MAP_TILES   = 'CartoDB positron'  # Folium tile layer
COLORMAP    = 'RdYlGn'           # Matplotlib colormap for choropleth

# ── Raster preview symbology ───────────────────────────────────
# Set RASTER_IS_BINARY = True if your raster contains only 0 and 1 values.
# You can then customise the colour for each class below.
# Set RASTER_IS_BINARY = False to use a continuous colormap instead.
RASTER_IS_BINARY = True           # <-- True for binary (0/1) rasters
RASTER_COLOR_0   = '#8c8c8c'      # <-- colour for value 0  (default: grey)
RASTER_COLOR_1   = '#27ae60'      # <-- colour for value 1  (default: green)
RASTER_LABEL_0   = 'Absent (0)'   # <-- legend label for 0
RASTER_LABEL_1   = 'Present (1)'  # <-- legend label for 1

print('✅ Configuration loaded.')
print(f'   Raster 2018 : {RASTER_2018}')
print(f'   Raster 2021 : {RASTER_2021}')
print(f'   Output dir  : {OUT_DIR.resolve()}')

In [ ]:
# ── Standard library ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show as rshow
from rasterio.mask import mask as rmask
from rasterstats import zonal_stats

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from folium.plugins import MeasureControl, MiniMap
import branca.colormap as bcm

# ── Notebook utilities ────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Global style ──────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

print('✅ All libraries imported.')

In [ ]:
# ================================================================
#  HELPER FUNCTIONS  (shared across all use cases)
# ================================================================

# ── 1. Load & reproject vector to raster CRS ─────────────────
def load_vector(path: Path, raster_path: Path) -> gpd.GeoDataFrame:
    """Load a GeoJSON/shapefile and reproject to the raster CRS."""
    gdf = gpd.read_file(path)
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
    if gdf.crs != raster_crs:
        print(f'  ↪ Reprojecting vector from {gdf.crs} to {raster_crs}')
        gdf = gdf.to_crs(raster_crs)
    return gdf


# ── 2. Run zonal statistics for one raster ───────────────────
def run_zonal_stats(
    gdf: gpd.GeoDataFrame,
    raster_path: Path,
    stats: list,
    band: int = 1,
    prefix: str = ''
) -> gpd.GeoDataFrame:
    """
    Compute zonal statistics for each polygon in gdf against a raster.
    Returns gdf enriched with stat columns prefixed by `prefix`.
    """
    results = zonal_stats(
        vectors=gdf,
        raster=str(raster_path),
        stats=stats,
        band=band,
        nodata=np.nan,
        geojson_out=False
    )
    stats_df = pd.DataFrame(results)
    if prefix:
        stats_df.columns = [f'{prefix}_{c}' for c in stats_df.columns]
    return gdf.reset_index(drop=True).join(stats_df)


# ── 3. Compute change between two years ──────────────────────
def compute_change(
    gdf: gpd.GeoDataFrame,
    stat: str = 'mean',
    year_a: str = '2018',
    year_b: str = '2021'
) -> gpd.GeoDataFrame:
    """
    Add absolute and relative change columns for a given statistic.
    Expects columns named <year_a>_<stat> and <year_b>_<stat>.
    """
    col_a = f'{year_a}_{stat}'
    col_b = f'{year_b}_{stat}'
    gdf[f'delta_{stat}']    = gdf[col_b] - gdf[col_a]
    gdf[f'pct_chg_{stat}']  = ((gdf[col_b] - gdf[col_a]) / gdf[col_a].abs()) * 100
    return gdf


# ── 4. Folium interactive overview map ───────────────────────
def make_overview_map(
    gdf: gpd.GeoDataFrame,
    id_col: str,
    title: str = 'AOI Overview'
) -> folium.Map:
    """Create a Folium map with polygons and popups."""
    gdf_wgs = gdf.to_crs(epsg=4326)
    centroid = gdf_wgs.geometry.unary_union.centroid
    m = folium.Map(
        location=[centroid.y, centroid.x],
        zoom_start=10,
        tiles=MAP_TILES
    )
    folium.GeoJson(
        gdf_wgs.__geo_interface__,
        name='AOI',
        style_function=lambda f: {
            'fillColor': '#3186cc',
            'color': '#ffffff',
            'weight': 1.5,
            'fillOpacity': 0.35
        },
        tooltip=folium.GeoJsonTooltip(fields=[id_col], aliases=['ID:'])
    ).add_to(m)
    MiniMap(toggle_display=True).add_to(m)
    MeasureControl().add_to(m)
    folium.LayerControl().add_to(m)
    return m


# ── 5. Choropleth change map ──────────────────────────────────
def make_choropleth_map(
    gdf: gpd.GeoDataFrame,
    value_col: str,
    id_col: str,
    title: str = 'Change Map',
    cmap: str = 'RdYlGn'
) -> folium.Map:
    """Choropleth map of a numeric column."""
    gdf_wgs = gdf.to_crs(epsg=4326).copy()
    gdf_wgs[id_col] = gdf_wgs[id_col].astype(str)
    centroid = gdf_wgs.geometry.unary_union.centroid
    m = folium.Map(
        location=[centroid.y, centroid.x],
        zoom_start=10,
        tiles=MAP_TILES
    )
    vals = gdf_wgs[value_col].dropna()
    vmin, vmax = vals.min(), vals.max()
    colorscale = bcm.linear.RdYlGn_11.scale(vmin, vmax)
    colorscale.caption = title
    colorscale.add_to(m)

    folium.GeoJson(
        gdf_wgs.__geo_interface__,
        name=title,
        style_function=lambda f: {
            'fillColor': colorscale(f['properties'].get(value_col) or vmin),
            'color': '#444',
            'weight': 0.8,
            'fillOpacity': 0.75
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[id_col, value_col],
            aliases=['ID:', f'{title}:'],
            localize=True
        )
    ).add_to(m)
    MiniMap(toggle_display=True).add_to(m)
    folium.LayerControl().add_to(m)
    return m


# ── 6. Side-by-side raster preview (binary or continuous) ────
def plot_raster_preview(
    raster_path_a: Path,
    raster_path_b: Path,
    aoi_gdf: gpd.GeoDataFrame = None,
    title_a: str = '2018',
    title_b: str = '2021',
    band: int = 1,
    is_binary: bool = None,
    color_0: str = None,
    color_1: str = None,
    label_0: str = None,
    label_1: str = None
):
    """
    Plot two rasters side by side, optionally overlaid with AOI boundaries.

    For binary rasters (0/1), pass is_binary=True (or rely on RASTER_IS_BINARY
    from the config cell).  Colours and legend labels are read from the config
    variables but can be overridden per-call via color_0/color_1/label_0/label_1.

    For continuous rasters, the COLORMAP variable is used.
    """
    # ── resolve defaults from config globals ─────────────────────
    if is_binary is None:
        is_binary = RASTER_IS_BINARY
    if color_0 is None:
        color_0 = RASTER_COLOR_0
    if color_1 is None:
        color_1 = RASTER_COLOR_1
    if label_0 is None:
        label_0 = RASTER_LABEL_0
    if label_1 is None:
        label_1 = RASTER_LABEL_1

    if is_binary:
        # Build a discrete 2-colour colormap  0 → color_0 / 1 → color_1
        binary_cmap = mcolors.ListedColormap([color_0, color_1])
        norm         = mcolors.BoundaryNorm([0, 0.5, 1], binary_cmap.N)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, path, title in zip(axes, [raster_path_a, raster_path_b], [title_a, title_b]):
        with rasterio.open(path) as src:
            data = src.read(band, masked=True)
            extent = [
                src.bounds.left, src.bounds.right,
                src.bounds.bottom, src.bounds.top
            ]

        if is_binary:
            im = ax.imshow(data, cmap=binary_cmap, norm=norm,
                           extent=extent, origin='upper', interpolation='nearest')
            # Custom legend patches instead of a colorbar
            patches = [
                mpatches.Patch(color=color_0, label=label_0),
                mpatches.Patch(color=color_1, label=label_1)
            ]
            ax.legend(handles=patches, loc='lower right',
                      fontsize=9, framealpha=0.85)
        else:
            im = ax.imshow(data, cmap=COLORMAP, extent=extent,
                           origin='upper')
            plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)

        if aoi_gdf is not None:
            aoi_gdf.boundary.plot(ax=ax, edgecolor='red', linewidth=0.8)

        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.axis('off')

    plt.suptitle('Raster Preview', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()


# ── 7. Summary statistics table ──────────────────────────────
def display_stats_table(
    gdf: gpd.GeoDataFrame,
    id_col: str,
    stat: str = 'mean'
):
    """Display a styled HTML comparison table."""
    cols = [id_col, f'2018_{stat}', f'2021_{stat}', f'delta_{stat}', f'pct_chg_{stat}']
    sub = gdf[cols].copy().round(3)
    sub.columns = [id_col, f'{stat.capitalize()} 2018', f'{stat.capitalize()} 2021',
                   'Δ Absolute', 'Δ %']

    def colour_delta(v):
        if pd.isna(v): return ''
        c = '#2ecc71' if v > 0 else '#e74c3c' if v < 0 else ''
        return f'color: {c}; font-weight: bold'

    styled = (
        sub.style
        .applymap(colour_delta, subset=['Δ Absolute', 'Δ %'])
        .format({'Δ %': '{:+.1f}%', 'Δ Absolute': '{:+.3f}'})
        .background_gradient(cmap='RdYlGn', subset=[f'{stat.capitalize()} 2018', f'{stat.capitalize()} 2021'])
        .set_table_styles([{'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white')]}])
        .set_caption(f'Zonal Statistics Comparison – {stat.upper()}')
    )
    display(styled)


# ── 8. Grouped bar chart ──────────────────────────────────────
def plot_bar_comparison(
    gdf: gpd.GeoDataFrame,
    id_col: str,
    stat: str = 'mean',
    title: str = 'Comparison',
    ylabel: str = 'Value',
    n: int = 20
):
    """Grouped bar chart comparing 2018 vs 2021 for top-n features."""
    sub = gdf[[id_col, f'2018_{stat}', f'2021_{stat}']].dropna().head(n)
    x = np.arange(len(sub))
    w = 0.38
    fig, ax = plt.subplots(figsize=(max(10, len(sub) * 0.55), 5))
    ax.bar(x - w/2, sub[f'2018_{stat}'], w, label='2018', color='#3498db', alpha=0.85)
    ax.bar(x + w/2, sub[f'2021_{stat}'], w, label='2021', color='#e67e22', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(sub[id_col].astype(str), rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()


# ── 9. Change distribution plot ─────────────────────────────
def plot_change_distribution(
    gdf: gpd.GeoDataFrame,
    stat: str = 'mean',
    title: str = 'Distribution of Change'
):
    """KDE + histogram of absolute change values."""
    delta_col = f'delta_{stat}'
    vals = gdf[delta_col].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    axes[0].hist(vals, bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
    axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_xlabel(f'Δ {stat}')
    axes[0].set_title('Distribution of Absolute Change')

    # Scatter 2018 vs 2021
    axes[1].scatter(gdf[f'2018_{stat}'], gdf[f'2021_{stat}'], alpha=0.6, edgecolors='grey', linewidths=0.3)
    lims = [
        min(gdf[f'2018_{stat}'].min(), gdf[f'2021_{stat}'].min()),
        max(gdf[f'2018_{stat}'].max(), gdf[f'2021_{stat}'].max())
    ]
    axes[1].plot(lims, lims, 'r--', linewidth=1, label='No change')
    axes[1].set_xlabel(f'{stat} 2018')
    axes[1].set_ylabel(f'{stat} 2021')
    axes[1].set_title('2018 vs 2021 Scatter')
    axes[1].legend()

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ── 10. Export function ───────────────────────────────────────
def export_results(
    gdf: gpd.GeoDataFrame,
    name: str,
    out_dir: Path = OUT_DIR
):
    """Export enriched GeoDataFrame as CSV and GeoJSON."""
    # Drop geometry for CSV
    df = pd.DataFrame(gdf.drop(columns='geometry'))
    csv_path  = out_dir / f'{name}_stats.csv'
    geoj_path = out_dir / f'{name}_stats.geojson'

    df.to_csv(csv_path, index=False)
    gdf.to_crs(epsg=4326).to_file(geoj_path, driver='GeoJSON')

    print(f'💾 Exported:')
    print(f'   CSV     → {csv_path}')
    print(f'   GeoJSON → {geoj_path}')
    return csv_path, geoj_path


# ── 11. Dataset-level summary table ──────────────────────────
def display_summary_table(
    gdf: gpd.GeoDataFrame,
    stat: str = 'mean'
):
    """
    Display a compact summary of key statistics across all polygons
    for both years and the change between them.
    Rows = metrics (mean, median, min, max, std, count).
    Columns = 2018 value / 2021 value / absolute change / % change.
    """
    col18, col21 = f'2018_{stat}', f'2021_{stat}'
    v18 = gdf[col18].dropna()
    v21 = gdf[col21].dropna()

    metrics = ['mean', 'median', 'min', 'max', 'std', 'count']
    rows = []
    for m in metrics:
        if m == 'count':
            val18 = v18.count()
            val21 = v21.count()
        else:
            val18 = getattr(v18, m)()
            val21 = getattr(v21, m)()
        delta    = val21 - val18
        pct      = (delta / abs(val18) * 100) if val18 != 0 else np.nan
        rows.append({'Metric': m.capitalize(), '2018': val18, '2021': val21,
                     'Δ Absolute': delta, 'Δ %': pct})

    summary = pd.DataFrame(rows).set_index('Metric').round(4)

    def colour_delta(v):
        if pd.isna(v): return ''
        c = '#2ecc71' if v > 0 else '#e74c3c' if v < 0 else ''
        return f'color: {c}; font-weight: bold'

    styled = (
        summary.style
        .applymap(colour_delta, subset=['Δ Absolute', 'Δ %'])
        .format({'Δ %': '{:+.2f}%', 'Δ Absolute': '{:+.4f}'}, na_rep='—')
        .background_gradient(cmap='Blues', subset=['2018', '2021'])
        .set_table_styles([{'selector': 'th', 'props': [
            ('background-color', '#2c3e50'), ('color', 'white'), ('font-size', '12px')
        ]}])
        .set_caption(f'Dataset Summary – zonal {stat.upper()} across all polygons')
    )
    display(styled)
    return summary


print('✅ Helper functions defined.')

---
<a id='usecaseA'></a>
# 🌾 Use Case A – Agriculture

**Objective:** Quantify raster-derived indicators (e.g. vegetation index, land cover value) over individual agricultural parcels and track their evolution between 2018 and 2021.

**Typical questions:**
- Which parcels showed the largest increase or decrease in the indicator?
- How did the distribution of values shift across the study area?

### A.0 Configuration

Update paths and parameters here. All cells below reference these variables.

### A.1 · Load Data

In [ ]:
print('📂 Loading agricultural parcels...')
agri_gdf = load_vector(AGRI_VECTOR, RASTER_2018)

print(f'   {len(agri_gdf)} parcels loaded')
print(f'   CRS   : {agri_gdf.crs}')
print(f'   Columns: {list(agri_gdf.columns)}')
agri_gdf.head()

### A.2 · Visualise Study Area

In [ ]:
# Interactive Folium overview map
agri_map = make_overview_map(agri_gdf, id_col=AGRI_ID_COL, title='Agricultural Parcels Overview')
display(agri_map)

In [ ]:
# Side-by-side raster preview clipped to AOI bounding box
plot_raster_preview(
    RASTER_2018, RASTER_2021,
    aoi_gdf=agri_gdf,
    title_a='Raster 2018 – Agriculture AOI',
    title_b='Raster 2021 – Agriculture AOI'
)

### A.3 · Zonal Statistics

In [ ]:
# Zonal statistics are run on the full dataset
agri_sub = agri_gdf.copy()

print('⏳ Computing zonal statistics for 2018...')
agri_sub = run_zonal_stats(agri_sub, RASTER_2018, ZONAL_STATS, band=BAND_INDEX, prefix='2018')

print('⏳ Computing zonal statistics for 2021...')
agri_sub = run_zonal_stats(agri_sub, RASTER_2021, ZONAL_STATS, band=BAND_INDEX, prefix='2021')

# Compute change
agri_sub = compute_change(agri_sub, stat='mean')
agri_sub = compute_change(agri_sub, stat='median')

print(f'✅ Zonal statistics complete — {len(agri_sub)} parcels processed.')

### A.4 · Results Summary

In [ ]:
# Dataset-level summary: key stats across all parcels for each year
agri_summary = display_summary_table(agri_sub, stat='mean')

### A.5 · Visualisations

In [ ]:
plot_bar_comparison(
    agri_sub,
    id_col=AGRI_ID_COL,
    stat='mean',
    title='Agriculture – Mean Raster Value per Parcel (2018 vs 2021)',
    ylabel='Mean Value'
)

In [ ]:
plot_change_distribution(
    agri_sub,
    stat='mean',
    title='Agriculture – Distribution of Change in Mean Value'
)

In [ ]:
# Choropleth map of absolute change
agri_change_map = make_choropleth_map(
    agri_sub,
    value_col='delta_mean',
    id_col=AGRI_ID_COL,
    title='Δ Mean Value 2018→2021 (Agriculture)'
)
display(agri_change_map)

In [ ]:
# Boxplot: distribution of mean values per year
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(
    [agri_sub['2018_mean'].dropna(), agri_sub['2021_mean'].dropna()],
    labels=['2018', '2021'],
    patch_artist=True,
    boxprops=dict(facecolor='#3498db', alpha=0.7),
    medianprops=dict(color='black', linewidth=2)
)
ax.set_ylabel('Mean Raster Value')
ax.set_title('Agriculture – Distribution of Parcel Mean Values')
plt.tight_layout()
plt.show()

### A.6 · Export Results

In [ ]:
csv_a, geoj_a = export_results(agri_sub, name='usecaseA_agriculture')

---
<a id='usecaseB'></a>
# 🦋 Use Case B – Biodiversity

**Objective:** Aggregate Woody Vegetation Layers indicators at administrative unit level, identifying spatial patterns and temporal trends between 2018 and 2021.

**Typical questions:**
- Which administrative units show the highest habitat quality?
- Where did conditions improve or deteriorate?


### B.0 . Configuration

if configuration comes here, the helper function cells should be duplicated after and run after config step

### B.1 · Load Data

In [ ]:
print('📂 Loading administrative units...')
admin_gdf = load_vector(ADMIN_VECTOR, RASTER_2018)

print(f'   {len(admin_gdf)} units loaded')
print(f'   CRS     : {admin_gdf.crs}')
print(f'   Columns : {list(admin_gdf.columns)}')
admin_gdf.head()

### B.2 · Visualise Study Area

In [ ]:
admin_map = make_overview_map(admin_gdf, id_col=ADMIN_ID_COL, title='Administrative Units Overview')
display(admin_map)

In [ ]:
plot_raster_preview(
    RASTER_2018, RASTER_2021,
    aoi_gdf=admin_gdf,
    title_a='Raster 2018 – Admin Units',
    title_b='Raster 2021 – Admin Units'
)

### B.3 · Zonal Statistics

In [ ]:
admin_sub = admin_gdf.copy()

print('⏳ Computing zonal statistics for 2018...')
admin_sub = run_zonal_stats(admin_sub, RASTER_2018, ZONAL_STATS, band=BAND_INDEX, prefix='2018')

print('⏳ Computing zonal statistics for 2021...')
admin_sub = run_zonal_stats(admin_sub, RASTER_2021, ZONAL_STATS, band=BAND_INDEX, prefix='2021')

admin_sub = compute_change(admin_sub, stat='mean')
admin_sub = compute_change(admin_sub, stat='median')

print('✅ Zonal statistics complete.')
admin_sub[[ADMIN_ID_COL, '2018_mean', '2021_mean', 'delta_mean', 'pct_chg_mean']].head()

### B.4 · Results Table

In [ ]:
display_stats_table(admin_sub, id_col=ADMIN_ID_COL, stat='mean')

### B.5 · Visualisations

In [ ]:
plot_bar_comparison(
    admin_sub,
    id_col=ADMIN_ID_COL,
    stat='mean',
    title='Biodiversity – Mean Value per Admin Unit (2018 vs 2021)',
    ylabel='Mean Value'
)

In [ ]:
# Side-by-side static choropleth maps
admin_wgs = admin_sub.to_crs(epsg=4326)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, col, year in zip(axes, ['2018_mean', '2021_mean'], ['2018', '2021']):
    admin_wgs.plot(
        column=col,
        cmap=COLORMAP,
        legend=True,
        ax=ax,
        edgecolor='grey',
        linewidth=0.4,
        missing_kwds={'color': 'lightgrey', 'label': 'No data'}
    )
    ax.set_title(f'Biodiversity – Mean Value {year}', fontsize=12, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'usecaseB_biodiversity_maps.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Change choropleth
biodiv_change_map = make_choropleth_map(
    admin_sub,
    value_col='delta_mean',
    id_col=ADMIN_ID_COL,
    title='Δ Mean Value 2018→2021 (Biodiversity)'
)
display(biodiv_change_map)

In [ ]:
# Heatmap: stats across admin units
stat_cols = [f'2018_{s}' for s in ['min', 'mean', 'median', 'max', 'std']] + \
            [f'2021_{s}' for s in ['min', 'mean', 'median', 'max', 'std']]
heatmap_df = admin_sub.set_index(ADMIN_ID_COL)[stat_cols].dropna(how='all')

fig, ax = plt.subplots(figsize=(14, max(4, len(heatmap_df) * 0.4)))
sns.heatmap(
    heatmap_df,
    cmap=COLORMAP,
    annot=len(heatmap_df) <= 20,
    fmt='.1f',
    linewidths=0.4,
    ax=ax
)
ax.set_title('Biodiversity – Zonal Statistics Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'usecaseB_biodiversity_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
plot_change_distribution(
    admin_sub,
    stat='mean',
    title='Biodiversity – Distribution of Change in Mean Value'
)

### B.7 · Export Results

In [ ]:
csv_b, geoj_b = export_results(admin_sub, name='usecaseB_biodiversity')

---
<a id='usecaseC'></a>
# 🏙️ Use Case C – Urban

**Objective:** Quantify raster-derived urban indicators (tree canopy extent) over urban extent polygons, comparing the 2018 and 2021 states to reveal patterns.

**Typical questions:**
- Which urban zones experienced the largest increase in imperviousness?
- Are there urban areas that showed vegetation recovery?


### C.0. Configuration

if configuration comes here, the helper function cells should be duplicated after and run after config step

### C.1 · Load Data

In [ ]:
print('📂 Loading urban areas...')
urban_gdf = load_vector(URBAN_VECTOR, RASTER_2018)

print(f'   {len(urban_gdf)} urban polygons loaded')
print(f'   CRS     : {urban_gdf.crs}')
print(f'   Columns : {list(urban_gdf.columns)}')
urban_gdf.head()

### C.2 · Visualise Study Area

In [ ]:
urban_map = make_overview_map(urban_gdf, id_col=URBAN_ID_COL, title='Urban Areas Overview')
display(urban_map)

In [ ]:
plot_raster_preview(
    RASTER_2018, RASTER_2021,
    aoi_gdf=urban_gdf,
    title_a='Raster 2018 – Urban Areas',
    title_b='Raster 2021 – Urban Areas'
)

### C.3 · Zonal Statistics

In [ ]:
urban_sub = urban_gdf.copy()

print('⏳ Computing zonal statistics for 2018...')
urban_sub = run_zonal_stats(urban_sub, RASTER_2018, ZONAL_STATS, band=BAND_INDEX, prefix='2018')

print('⏳ Computing zonal statistics for 2021...')
urban_sub = run_zonal_stats(urban_sub, RASTER_2021, ZONAL_STATS, band=BAND_INDEX, prefix='2021')

urban_sub = compute_change(urban_sub, stat='mean')
urban_sub = compute_change(urban_sub, stat='median')

print('✅ Zonal statistics complete.')
urban_sub[[URBAN_ID_COL, '2018_mean', '2021_mean', 'delta_mean', 'pct_chg_mean']].head()

### C.4 · Results Table

In [ ]:
display_stats_table(urban_sub, id_col=URBAN_ID_COL, stat='mean')

### C.5 · Visualisations

In [ ]:
plot_bar_comparison(
    urban_sub,
    id_col=URBAN_ID_COL,
    stat='mean',
    title='Urban – Mean WVL Value per Zone (2018 vs 2021)',
    ylabel='Mean Value'
)

In [ ]:
plot_change_distribution(
    urban_sub,
    stat='mean',
    title='Urban – Distribution of Change in Mean Value'
)

In [ ]:
# Change choropleth
urban_change_map = make_choropleth_map(
    urban_sub,
    value_col='delta_mean',
    id_col=URBAN_ID_COL,
    title='Δ Mean Value 2018→2021 (Urban)'
)
display(urban_change_map)

In [ ]:
# Top-10 areas by absolute change
top_change = urban_sub.nlargest(10, 'delta_mean')[[URBAN_ID_COL, 'delta_mean', 'pct_chg_mean']]
bot_change = urban_sub.nsmallest(10, 'delta_mean')[[URBAN_ID_COL, 'delta_mean', 'pct_chg_mean']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, label, color in zip(
    axes,
    [top_change, bot_change],
    ['Top 10 Increase', 'Top 10 Decrease'],
    ['#2ecc71', '#e74c3c']
):
    ax.barh(df[URBAN_ID_COL].astype(str), df['delta_mean'], color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Δ Mean Value')
    ax.set_title(f'{label}', fontsize=12, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Urban Areas – Extremes of Change (2018→2021)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'usecaseC_urban_top_change.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Violin plot
plot_data = pd.DataFrame({
    'Value': pd.concat([urban_sub['2018_mean'], urban_sub['2021_mean']]),
    'Year': ['2018'] * len(urban_sub) + ['2021'] * len(urban_sub)
})
fig, ax = plt.subplots(figsize=(7, 4))
sns.violinplot(data=plot_data, x='Year', y='Value', palette=['#3498db', '#e67e22'], ax=ax)
ax.set_title('Urban – Value Distribution per Year', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### C.7 · Export Results

In [ ]:
csv_c, geoj_c = export_results(urban_sub, name='usecaseC_urban')

---
## 📊 Cross-Use-Case Summary

Quick overview of the mean delta across all three use cases.

In [ ]:
summary = pd.DataFrame({
    'Use Case': ['A – Agriculture', 'B – Biodiversity', 'C – Urban'],
    'N Features': [len(agri_sub), len(admin_sub), len(urban_sub)],
    'Mean 2018': [
        agri_sub['2018_mean'].mean(),
        admin_sub['2018_mean'].mean(),
        urban_sub['2018_mean'].mean()
    ],
    'Mean 2021': [
        agri_sub['2021_mean'].mean(),
        admin_sub['2021_mean'].mean(),
        urban_sub['2021_mean'].mean()
    ],
    'Mean Δ': [
        agri_sub['delta_mean'].mean(),
        admin_sub['delta_mean'].mean(),
        urban_sub['delta_mean'].mean()
    ],
    'Mean % Δ': [
        agri_sub['pct_chg_mean'].mean(),
        admin_sub['pct_chg_mean'].mean(),
        urban_sub['pct_chg_mean'].mean()
    ]
}).set_index('Use Case').round(3)

display(summary.style
    .background_gradient(cmap='RdYlGn', subset=['Mean Δ', 'Mean % Δ'])
    .format({'Mean % Δ': '{:+.1f}%', 'Mean Δ': '{:+.3f}'})
    .set_caption('Cross-Use-Case Summary of Temporal Change')
)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in summary['Mean Δ']]
summary['Mean Δ'].plot(kind='bar', ax=ax, color=colors, edgecolor='white', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Absolute Change (2018→2021)')
ax.set_title('Cross-Use-Case: Mean Change in Raster Value', fontsize=12, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'summary_cross_usecase.png', bbox_inches='tight', dpi=150)
plt.show()

# Save summary CSV
summary.to_csv(OUT_DIR / 'summary_cross_usecase.csv')
print(f'💾 Summary saved to {OUT_DIR / "summary_cross_usecase.csv"}')

---

## ✅ All Done!

All outputs have been saved to the `outputs/` directory:

| File | Content |
|------|---------|
| `usecaseA_agriculture_stats.csv` | Zonal stats per parcel |
| `usecaseA_agriculture_stats.geojson` | Enriched parcel polygons |
| `usecaseB_biodiversity_stats.csv` | Zonal stats per admin unit |
| `usecaseB_biodiversity_stats.geojson` | Enriched admin polygons |
| `usecaseC_urban_stats.csv` | Zonal stats per urban area |
| `usecaseC_urban_stats.geojson` | Enriched urban polygons |
| `summary_cross_usecase.csv` | Aggregated cross-use-case summary |
